## Задание 1

Реализуйте класс с полностью инкапсулированным состоянием, используя name mangling и property, обеспечив валидацию при изменении атрибутов и демонстрируя, как Python скрывает "приватные" данные.

In [22]:
class Encapsulated:
    def __init__(self, value):
        self.__value = None
        self.value = value

    @property
    def value(self):
        return self.__value

    @value.setter
    def value(self, new_value):
        if not isinstance(new_value, (int, float)):
            raise TypeError("value должен быть числом")
        self.__value = new_value


obj = Encapsulated(10)
print(obj.value)
print(obj._Encapsulated__value)

obj.value = 3
print(obj.value)
print(obj._Encapsulated__value)

10
10
3
3


In [23]:
"""
так как в сеттере стоит проверка на тип и значение, то присвоение к value объектов
кроме (int, float)
"""

obj.value = 'hi'

TypeError: value должен быть числом

## Задание 2

Создайте иерархию классов с абстрактным базовым классом (ABC) и абстрактными методами, демонстрируя использование модуля abc. Реализуйте интерфейсы и их проверку через isinstance и issubclass.

In [ ]:
from abc import ABC, abstractmethod


class Animal(ABC):
    @abstractmethod
    def sound(self):
        pass

    @abstractmethod
    def move(self):
        pass


class Dog(Animal):
    def sound(self):
        return "Dog_sound"

    def move(self):
        return "Dog_move"


class Cat(Animal):
    def sound(self):
        return "Cat_sound"

    def move(self):
        return "Cat_move"


dog = Dog()
cat = Cat()

print(dog.sound(), "-", dog.move())
print(cat.sound(), "-", cat.move())
print(isinstance(dog, Animal))
print(issubclass(Dog, Animal))

Dog_sound - Dog_move
Cat_sound - Cat_move
True
True


## Задание 3

Реализуйте дженерик-функцию (duck typing) с использованием протоколов (PEP 544) и типовых подсказок, чтобы показать полиморфизм без наследования.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> None:
        pass
    

class Circle:
    def draw(self) -> None:
        print("draw circle")


class Square:
    def draw(self) -> None:
        print("square circle")


def paint(obj: Drawable) -> None:
    obj.draw()


circle = Circle()
square = Square()

paint(circle)
paint(square)

print(isinstance(circle, Drawable))
print(isinstance(square, Drawable))

draw circle
square circle
True
True


## Задание 4

Опишите и продемонстрируйте работу метода getattribute в отличие от getattr, реализуйте логирование всех обращений к атрибутам объекта, исследуйте возможные зацикливания.

In [24]:
class Logger:
    def __init__(self):
        object.__setattr__(self, "_log", [])

    def __getattribute__(self, name):
        if name not in {"_log", "__dict__", "__class__"}:
            print(f"__getattribute__ -> {name}")
            object.__getattribute__(self, "_log").append(name)
        return object.__getattribute__(self, name)

    def __getattr__(self, name):
        print(f"__getattr__ -> {name} не найден")
        return f"<атрибут {name} отсутствует>"


obj = Logger()
obj.x = 5

print("Обращаемся к obj.x")
print(obj.x)
print("\n-------------------\n")
print("Обращаемся к obj.y")
print(obj.y)
print("\n-------------------\n")
print("Обращаемся к obj._log")
print(obj._log)

Обращаемся к obj.x
__getattribute__ -> x
5

-------------------

Обращаемся к obj.y
__getattribute__ -> y
__getattr__ -> y не найден
<атрибут y отсутствует>

-------------------

Обращаемся к obj._log
['x', 'y']


## Задание 5

Создайте класс, который использует метакласс, объясните как метаклассы влияют на создание классов в Python, и реализуйте контролируемое изменение класса через метакласс.

In [30]:
class Meta(type):
    def __new__(mcs, name, bases, namespace):
        namespace["created_by_meta"] = True

        if "hello" not in namespace:
            def hello(self):
                return f"Привет от {self.__class__.__name__}"
            namespace["hello"] = hello

        cls = super().__new__(mcs, name, bases, namespace)
        cls.version = "1.0"
        return cls


class ClassThatHaveMeta(metaclass=Meta):
    pass


obj = ClassThatHaveMeta()
print(obj.hello())
print(ClassThatHaveMeta.created_by_meta)
print(ClassThatHaveMeta.version)

Привет от ClassThatHaveMeta
True
1.0


## Задание 6

Реализуйте класс с дескрипторами данных и неданных, объясните разницу между ними и механизм вызова методов get, set, delete, особенно при наследовании.

In [31]:
class DataDescriptor:
    def __set_name__(self, owner, name):
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return "DataDescriptor на уровне класса"
        return instance.__dict__.get(self.storage_name, "значение не установлено")

    def __set__(self, instance, value):
        print(f"DataDescriptor.__set__ -> {value}")
        instance.__dict__[self.storage_name] = value

    def __delete__(self, instance):
        instance.__dict__.pop(self.storage_name, None)


class NonDataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return "NonDataDescriptor на уровне класса"
        return "значение из non-data descriptor"


class MyClass2:
    data_desc = DataDescriptor()
    non_data_desc = NonDataDescriptor()


obj = MyClass2()
obj.data_desc = 10
print(obj.data_desc)

obj.non_data_desc = "значение из экземпляра"
print(obj.non_data_desc)
print(MyClass2.non_data_desc)

DataDescriptor.__set__ -> 10
10
значение из экземпляра
NonDataDescriptor на уровне класса


## Задание 7

Напишите класс с поддержкой множественного наследования, демонстрирующий работу C3-линеаризации MRO на сложном примере с 3+ уровнями наследования и пересечениями.

In [32]:
class A:
    def method(self):
        print("A.method")


class B(A):
    def method(self):
        print("B.method")
        super().method()


class C(A):
    def method(self):
        print("C.method")
        super().method()


class D(B, C):
    def method(self):
        print("D.method")
        super().method()


d = D()
d.method()
print(D.__mro__)

D.method
B.method
C.method
A.method
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


## Задание 8

Реализуйте класс с пользовательскими dunder-методами: new, init, call, del, repr, str, и объясните, как Python вызывает их в цикле жизни объекта.

In [37]:
class LifeCycle:
    def __new__(cls, *args, **kwargs):
        print("__new__: создание объекта")
        return super().__new__(cls)

    def __init__(self):
        print("__init__: инициализация объекта")
        self.state = "ready"

    def __call__(self):
        print("__call__: объект вызван как функция")

    def __repr__(self):
        return "LifeCycle(state='ready')"

    def __str__(self):
        return "Экземпляр LifeCycle"

    def __del__(self):
        print("__del__: объект удаляется")


obj = LifeCycle()
print(obj)
print(repr(obj))
obj()
del obj

__new__: создание объекта
__init__: инициализация объекта
Экземпляр LifeCycle
LifeCycle(state='ready')
__call__: объект вызван как функция
__del__: объект удаляется


## Задание 9

Создайте контекстный менеджер с помощью специальных методов enter и exit, используйте его вместе с классом, в котором присутствуют методы с разграничением прав доступа.

https://habr.com/ru/articles/739326/

In [40]:
class Access:
    def __init__(self):
        self._secret = "Super secrete data"
        self._access_granted = False

    def __enter__(self):
        self._access_granted = True
        return self._secret

    def __exit__(self, exc_type, exc_val, exc_tb):
        self._access_granted = False
        return False

    @property
    def secret(self):
        if self._access_granted:
            return self._secret
        return "Доступ запрещён"


obj = Access()
print(obj.secret)
with obj as secret_value:
    print(secret_value)
    print(obj.secret)
print(obj.secret)

Доступ запрещён
Super secrete data
Super secrete data
Доступ запрещён


## Задание 10

Создайте класс, объекты которого могут быть отслежены с помощью слабых ссылок (weakref). Реализуйте систему, которая хранит слабые ссылки на все созданные объекты класса и автоматически удаляет их из списка при уничтожении объектов. Продемонстрируйте это поведение, выводя текущее количество живых объектов.

In [ ]:
import weakref


class Tracked:
    _refs = []

    def __init__(self, name):
        self.name = name

        def _remove(dead_ref):
            Tracked._refs = [ref for ref in Tracked._refs if ref is not dead_ref]
            print(f"Объект удалён. Живых объектов: {Tracked.alive_count()}")

        Tracked._refs.append(weakref.ref(self, _remove))

    @classmethod
    def alive_count(cls):
        cls._refs = [ref for ref in cls._refs if ref() is not None]
        return len(cls._refs)

    def __repr__(self):
        return f"Tracked({self.name!r})"


a = Tracked("A")
b = Tracked("B")
c = Tracked("C")
print("Живых объектов:", Tracked.alive_count())

del b
print("После удаления b:", Tracked.alive_count())

del a
del c
print("После удаления всех объектов:", Tracked.alive_count())

Живых объектов: 3
Объект удалён. Живых объектов: 2
После удаления b: 2
Объект удалён. Живых объектов: 1
Объект удалён. Живых объектов: 0
После удаления всех объектов: 0
